# Test polytopes' volume computation

## Librairies

In [2]:
%load_ext autoreload
%autoreload 2

import sys
import os

# Go to project root (adjust depth if needed)
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(ROOT, "data")
sys.path.append(ROOT)

print("Added to path:", ROOT)

import torch
from torchvision import datasets, transforms
from torch.utils.data import Subset

from src.models.networks import FashionMLP_Large
from src.quantization.quantize import quantize_model
from src.shortcuts.shortcut_weights import *
from src.optim.build_polytopes import *
from src.optim.prune_constraints import *
from src.optim.compute_volumes import estimate_polytope_width


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Added to path: /Users/jeremiecabessa/Desktop/ROOT/Articles/Conference_Papers/2025_With_Jiri/code/ErrorVolumePolytopes


In [3]:
torch.manual_seed(42)

## Large MLP (FashionMNIST)

In [4]:
# -------------------- #
# Load models and data #
# -------------------- #

# Device
device = torch.device("gpu" if torch.cuda.is_available() 
                      else "mps" if torch.backends.mps.is_available() 
                      else "cpu")

# Model
model_name = "fashion_mlp_best.pth"
model_path = os.path.join(ROOT, "checkpoints", model_name)

model = FashionMLP_Large()
model.load_state_dict(torch.load(model_path))#['model_state'])
model.to(device).eval()

# qModel
qmodel = quantize_model(model, bits=4)
qmodel.to(device).eval()

# Dataset (and sample)
train_dataset = torch.load(
    DATA_PATH + "/fashionMNIST_correct_mlp.pt",
    weights_only=False
)

x_0, c = train_dataset[123]
print(x_0.min().item(), x_0.max().item()) # check bounds
x_0 = x_0.flatten().unsqueeze(0) # shape (1, input_dim)
x_0 = x_0.to(device) # important
print("Sample x_0 shape:", x_0.shape)
print("Models and dataset have been loaded.")

-1.0 1.0
Sample x_0 shape: torch.Size([1, 784])
Models and dataset have been loaded.


In [ ]:
# --------------------------- #
# Compute volume of polytopes #
# --------------------------- #

print("*** Computing volumes of polytopes... ***\n\n")

nb_directions = 20

bounds = [(-1., 1.)]

A_correct, b_correct, A_both, b_both = build_two_class_polytopes(model, qmodel, x_0, c)

results = estimate_polytope_width(A_correct, b_correct, bounds, 
                                    n_directions=nb_directions,
                                    tol=1e-9, verbose=True)
print("Mean polytope width:", results[0])
print("Std polytope width:", results[1])

*** Computing volumes of polytopes... ***


LP success at direction 0 width: 38.90991931489896
LP success at direction 1 width: 38.585985889917865
LP success at direction 2 width: 38.11354264031392
LP success at direction 3 width: 38.63212082725997
LP success at direction 4 width: 38.04760866560831
LP success at direction 5 width: 37.89024648878543
